# PatchTST — Walmart Store Sales Forecasting

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ["WALMART_ROOT"] = "/content/drive/MyDrive/MLFinalAssignment"

In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'PatchTST_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

In [ ]:
import time

import numpy as np
import pandas as pd

import torch
import mlflow

from walmart_panel import WalmartPanel
from walmart_dl import PatchTST, train_torch, SeqForecaster

from walmart_prep import (
    FOLDS, HORIZON, MIN_SAFE_LAG,
    WalmartFeatureBuilder, december_shape_report, load_raw, make_submission,
    score_fold, seasonal_naive, wmae_weights,
)

print("tracking:", mlflow.get_tracking_uri())

train, test, features, stores = load_raw(str(ROOT))
SEED = 0

print(f"train {train.shape}   test {test.shape}")
print(f"train {train.Date.min().date()} .. {train.Date.max().date()}")
print(f"test  {test.Date.min().date()} .. {test.Date.max().date()}")
print(f"horizon = {HORIZON} weeks  ->  minimum safe lag = {MIN_SAFE_LAG} weeks")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}  torch={torch.__version__}")
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

## `PatchTST_Cleaning`

In [ ]:
with mlflow.start_run(run_name="PatchTST_Cleaning"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({
        "panel": "dense [n_series x n_weeks], internal gaps -> 0",
        "normalisation": "per-series z-score from training weeks only",
        "loss": "WMAE-weighted L1 (holiday horizon steps x5), in normalised space",
        "target_modes": "level (raw series) and residual (YoY change over lag-52 baseline)",
    })
    supply = []
    for L in (52, 39):
        for fold in list(FOLDS) + [None]:
            cut = None if fold is None else fold.cut
            name = "FULL" if fold is None else fold.name
            try:
                p = WalmartPanel(train, features, stores, lookback=L).fit(cut)
                n = len(p.make_windows())
                supply.append({"lookback": L, "fold": name, "weeks": p.n_weeks_, "windows": n})
            except ValueError:
                supply.append({"lookback": L, "fold": name, "weeks": None, "windows": 0})
    supply = pd.DataFrame(supply)
    for _, r in supply.iterrows():
        mlflow.log_metric(f"windows_L{r.lookback}_{r.fold}", r.windows)
supply.pivot(index="fold", columns="lookback", values="windows")

## `PatchTST_Baseline`

In [ ]:
baseline_scores = {}
with mlflow.start_run(run_name="PatchTST_Baseline"):
    mlflow.log_param("model", "seasonal naive: lag-52, fallback pair median, then global median")
    for fold in FOLDS:
        tr = train[train.Date <= fold.cut]
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        s = score_fold(va.Weekly_Sales, seasonal_naive(tr, va), va)
        baseline_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])

BASELINE = {k: v["wmae"] for k, v in baseline_scores.items()}
pd.DataFrame(baseline_scores).round(1)

## `PatchTST_CV`

In [ ]:
EPOCHS = 60


def make_net(panel, cfg):
    d = int(cfg["d_model"])
    return PatchTST(panel.lookback, panel.horizon, patch_len=int(cfg["patch_len"]),
                    stride=int(cfg["stride"]), d_model=d, n_heads=4,
                    n_layers=int(cfg["n_layers"]), d_ff=2 * d, dropout=float(cfg["dropout"]))


def fit_eval(fold, cfg, panel_cache):
    """Train one net on one fold with one config; return its fold scores."""
    key = (int(cfg["lookback"]), cfg["target_mode"], fold.name)
    if key not in panel_cache:
        panel_cache[key] = WalmartPanel(train, features, stores, lookback=int(cfg["lookback"]),
                                        target_mode=cfg["target_mode"]).fit(fold.cut)
    panel = panel_cache[key]
    net = train_torch(make_net(panel, cfg), panel, epochs=int(cfg["epochs"]),
                      lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]),
                      batch_size=1024, device=DEVICE, seed=SEED, verbose=False)
    va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
    s = score_fold(va.Weekly_Sales, SeqForecaster(panel, net)(va), va)
    return net, panel, s


PANELS = {}
DEFAULT_CFG = dict(lookback=52, target_mode="residual", patch_len=8, stride=4,
                   d_model=64, n_layers=2, dropout=0.1, lr=1e-3,
                   weight_decay=1e-2, epochs=60)

cv_rows = []
with mlflow.start_run(run_name="PatchTST_CV"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params(DEFAULT_CFG)
    trials = [("recent", 52), ("recent", 39), ("mirror", 39)]
    for fold_name, L in trials:
        fold = next(f for f in FOLDS if f.name == fold_name)
        cfg = {**DEFAULT_CFG, "lookback": L, "target_mode": "level"}
        _, _, s = fit_eval(fold, cfg, PANELS)
        row = {"fold": fold_name, "lookback": L, "wmae": s["wmae"],
               "naive": BASELINE[fold_name],
               "vs_naive_%": 100 * (1 - s["wmae"] / BASELINE[fold_name])}
        cv_rows.append(row)
        mlflow.log_metric(f"wmae_{fold_name}_L{L}", s["wmae"])
        print(f"level  {fold_name:7s} L={L}: WMAE={s['wmae']:7.1f}  "
              f"naive={BASELINE[fold_name]:7.1f}  ({row['vs_naive_%']:+.1f}%)")

pd.DataFrame(cv_rows).round(1)

## `PatchTST_TargetStudy`

In [ ]:
fold = next(f for f in FOLDS if f.name == "recent")
va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]

study = []
with mlflow.start_run(run_name="PatchTST_TargetStudy"):
    mlflow.log_param("device", DEVICE)
    for mode, dp in [("level", 0.1), ("residual", 0.1), ("residual", 0.0)]:
        cfg = {**DEFAULT_CFG, "target_mode": mode, "dropout": dp}
        _, _, s = fit_eval(fold, cfg, PANELS)
        study.append({"target_mode": mode, "dropout": dp, "wmae": s["wmae"],
                      "vs_naive_%": 100 * (1 - s["wmae"] / BASELINE["recent"])})
        mlflow.log_metric(f"wmae_{mode}_dp{dp}", s["wmae"])
        print(f"{mode:8s} dropout={dp}: WMAE={s['wmae']:7.1f}  "
              f"({study[-1]['vs_naive_%']:+.1f}% vs naive)")

pd.DataFrame(study).round(1)

## `PatchTST_Tuning`

In [ ]:
rng = np.random.default_rng(0)
GRID = {
    "lookback": [39, 52],
    "target_mode": ["level", "residual"],
    "patch_len": [4, 8],
    "stride": [2, 4],
    "d_model": [64, 128],
    "n_layers": [2, 3],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 1e-3],
    "weight_decay": [1e-4, 1e-2, 1e-1],
    "epochs": [60],
}
N_TRIALS = 8
fold = next(f for f in FOLDS if f.name == "recent")
va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]


def sample_cfg():
    return {k: (type(v[0])(rng.choice(v)) if not isinstance(v[0], str) else str(rng.choice(v)))
            for k, v in GRID.items()}


candidates = [("default", dict(DEFAULT_CFG))]
candidates += [(f"trial_{i:02d}", sample_cfg()) for i in range(N_TRIALS)]

trials = []
with mlflow.start_run(run_name="PatchTST_Tuning"):
    mlflow.log_params({"n_trials": N_TRIALS, "search": "random + incumbent",
                       "selection_fold": "recent", "device": DEVICE})
    for name, cfg in candidates:
        t0 = time.time()
        _, _, s = fit_eval(fold, cfg, PANELS)
        cfg = dict(cfg); cfg["wmae"] = s["wmae"]
        trials.append({"name": name, **cfg})
        with mlflow.start_run(run_name=f"PatchTST_Tuning__{name}", nested=True):
            mlflow.log_params({k: v for k, v in cfg.items() if k != "wmae"})
            mlflow.log_metric("wmae_recent", s["wmae"])
            mlflow.log_metric("fit_seconds", time.time() - t0)
        print(f"{name:10s} WMAE={s['wmae']:7.1f}  ({time.time() - t0:4.0f}s)  "
              + "  ".join(f"{k}={v}" for k, v in cfg.items() if k not in ("wmae",)))

    best = min(trials, key=lambda t: t["wmae"])
    mlflow.log_params({f"best_{k}": v for k, v in best.items() if k not in ("wmae", "name")})
    mlflow.log_metric("best_wmae_recent", best["wmae"])

print(f"\nbest: {best['name']}  WMAE={best['wmae']:.1f}  naive={BASELINE['recent']:.1f}")
best

## `PatchTST_Final`

In [ ]:
best_cfg = {k: v for k, v in best.items() if k not in ("wmae", "name")}
panel_full = WalmartPanel(train, features, stores, lookback=int(best_cfg["lookback"]),
                          target_mode=best_cfg["target_mode"]).fit()
net_full = train_torch(make_net(panel_full, best_cfg), panel_full,
                       epochs=int(best_cfg["epochs"]), lr=float(best_cfg["lr"]),
                       weight_decay=float(best_cfg["weight_decay"]), batch_size=1024,
                       device=DEVICE, seed=SEED, verbose=True)
forecaster = SeqForecaster(panel_full, net_full)

with mlflow.start_run(run_name="PatchTST_Final") as final_run:
    mlflow.log_param("device", DEVICE)
    mlflow.log_params(best_cfg)
    mlflow.log_metric("cv_wmae_recent", best["wmae"])

    y_pred = forecaster(test)
    print(f"predictions: n={len(y_pred)} mean={y_pred.mean():.1f} "
          f"min={y_pred.min():.1f} max={y_pred.max():.1f}")

    g = (WalmartFeatureBuilder(features, stores)
         .fit(train.drop(columns=["Weekly_Sales"]), train.Weekly_Sales).xmas_profile_)
    frame, verdict = december_shape_report(train, test, y_pred, g)
    print(frame.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"\npredicted peak {verdict['peak_predicted']} | implied {verdict['peak_implied']}")
    print("GATE:", "PASS" if verdict["passed"] else "REVIEW -- " + "; ".join(verdict["problems"]))
    mlflow.log_metric("december_xmas_gap_pct", verdict["xmas_gap_pct"])
    mlflow.log_metric("december_gate_passed", int(verdict["passed"]))

    logged_model = mlflow.pyfunc.log_model(
        name="model", python_model=forecaster,
        code_paths=[str(ROOT / "src" / "walmart_prep.py"),
                    str(ROOT / "src" / "walmart_panel.py"),
                    str(ROOT / "src" / "walmart_dl.py")],
        input_example=test.head(3),
    )
    MODEL_URI = logged_model.model_uri

    sub = make_submission(test, y_pred, ROOT / "submissions" / "patchtst.csv")
    mlflow.log_artifact(str(ROOT / "submissions" / "patchtst.csv"))
    FINAL_RUN_ID = final_run.info.run_id

print(f"\nrun_id = {FINAL_RUN_ID}\nmodel_uri = {MODEL_URI}")
sub.head()

In [ ]:
from mlflow import MlflowClient

loaded = mlflow.pyfunc.load_model(MODEL_URI)
reloaded = np.asarray(loaded.predict(test))
print("max |reloaded - original| =", float(np.abs(reloaded - y_pred).max()))
assert np.allclose(reloaded, y_pred, atol=1e-3), "the logged pyfunc must reproduce its predictions"
print("pyfunc round-trip OK")

mv = mlflow.register_model(MODEL_URI, REGISTERED_MODEL_NAME)
print(f"registered {mv.name} version {mv.version}")
try:
    MlflowClient().set_registered_model_alias(mv.name, "patchtst", mv.version)
    print(f"alias 'patchtst' -> version {mv.version}")
except Exception as e:
    print("alias not set (optional):", str(e)[:80])

In [ ]:
print("experiments: https://dagshub.com/smama23/MLFinalProject/experiments")
print("registry   : https://dagshub.com/smama23/MLFinalProject/models")
try:
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
    cols = [c for c in ("tags.mlflow.runName", "metrics.wmae_recent", "metrics.cv_wmae_recent",
                        "metrics.december_gate_passed") if c in runs.columns]
    summary = runs[cols] if cols else runs
    try:
        display(summary)
    except NameError:
        print(summary.to_string(index=False))
except Exception as e:
    print("could not fetch run summary (model is already logged):", e)